In [1]:
#Import libraries and load data
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from category_encoders.target_encoder import TargetEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [2]:
transactions = pd.read_csv('transactions_fe_v2.csv')
transactions.head()

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,has_error,error_count,verification_failure,balance_issue,risky_category_fast,new_category_high_value,cred_guessing_pattern,is_online_txn,online_verification_failure,online_cred_guess_high_value
0,15626211,2022-01-01 00:00:10,24,2827,chip transaction,14528,polokwane,limpopo,700.0,5499.0,...,0,0,0,0,0,0,0,0,0,0
1,17529598,2022-01-01 00:00:21,1326,5561,chip transaction,94123,east london,eastern cape,5200.0,5310.0,...,0,0,0,0,0,0,0,0,0,0
2,15750655,2022-01-01 00:00:21,696,4207,swipe transaction,46284,bloemfontein,free state,9300.0,5411.0,...,0,0,0,0,0,0,0,0,0,0
3,18999635,2022-01-01 00:00:29,1640,4967,swipe transaction,81536,port elizabeth,eastern cape,6000.0,5310.0,...,0,0,0,0,0,0,0,0,0,0
4,18403665,2022-01-01 00:01:10,185,4046,chip transaction,22204,kimberley,northern cape,8300.0,5541.0,...,0,0,0,0,0,0,0,0,0,0


# Feature Selection

In [14]:
LEAKAGE_COLS = [
    "id", "id.1", "id_y",
    "user_id", "card_id", "client_id_y",
    "card_number", "cvv", "expires",
    "address", "zip", 'merchant_id',
    "description", "errors", "gender",  # raw text already engineered
]

df_model = transactions.drop(columns=[c for c in LEAKAGE_COLS if c in transactions.columns])

In [4]:
df_model.columns

Index(['date', 'use_chip', 'merchant_city', 'merchant_state', 'mcc',
       'is_fraud', 'card_brand', 'card_type', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'per_capita_income', 'yearly_income', 'total_debt',
       'credit_score', 'num_credit_cards', 'is_negative_amount', 'abs_amount',
       'transaction_hour', 'transaction_dayofweek', 'is_weekend', 'is_night',
       'is_rush_hour', 'time_since_last_txn', 'user_avg_amount',
       'user_std_amount', 'amount_to_avg_ratio', 'txn_count_24h',
       'txn_count_7d', 'card_age_days', 'is_high_value', 'user_hour_mean',
       'user_hour_std', 'unusual_hour', 'hour_median', 'robust_z_hour',
       'unusual_hour_flag', 'is_iqr_outlier', 'is_z_outlier',
       'income_ratio_outlier', 'user_txn_count_24h', 'user_txn_count_7d',
       'user_amt_sum_24h', 'user_avg_amt_24h', 'is_first_merchant_vi

In [15]:
df_model.head()

,date,use_chip,merchant_city,merchant_state,mcc,is_fraud,card_brand,card_type,has_chip,num_cards_issued,...,has_error,error_count,verification_failure,balance_issue,risky_category_fast,new_category_high_value,cred_guessing_pattern,is_online_txn,online_verification_failure,online_cred_guess_high_value
0,2022-01-01 00:00:10,chip transaction,polokwane,limpopo,5499.0,0,Visa,Debit,YES,2,...,0,0,0,0,0,0,0,0,0,0
1,2022-01-01 00:00:21,chip transaction,east london,eastern cape,5310.0,0,Visa,Debit,YES,2,...,0,0,0,0,0,0,0,0,0,0
2,2022-01-01 00:00:21,swipe transaction,bloemfontein,free state,5411.0,0,Visa,Debit,NO,2,...,0,0,0,0,0,0,0,0,0,0
3,2022-01-01 00:00:29,swipe transaction,port elizabeth,eastern cape,5310.0,0,Visa,Debit,YES,1,...,0,0,0,0,0,0,0,0,0,0
4,2022-01-01 00:01:10,chip transaction,kimberley,northern cape,5541.0,0,Visa,Debit,YES,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
'''df_model["use_chip"] = (
    df_model["use_chip"]
    .map({"online transaction": 0, "chip transaction": 1, "swipe transaction": 2})
    .astype("int8")
)

df_model["card_brand"] = (
    df_model["card_brand"]
    .map({"Visa": 0, "Mastercard": 1, "Discovery": 2, "American Express": 3})
    .astype("int8")
)

df_model["has_chip"] = (
    df_model["has_chip"]
    .map({"NO": 0, "YES": 1})
    .astype("int8")
)

df_model["card_type"] = (
    df_model["card_type"]
    .map({"Debit": 0, "Prepaid": 1, "Credit": 2})
    .astype("int8")
)'''

In [4]:
df_model.isna().sum().sort_values(ascending=False)

date                            0
use_chip                        0
merchant_city                   0
merchant_state                  0
mcc                             0
                               ..
new_category_high_value         0
cred_guessing_pattern           0
is_online_txn                   0
online_verification_failure     0
online_cred_guess_high_value    0
Length: 87, dtype: int64

In [16]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["use_chip", "card_brand", "card_type"]

# Fit encoder
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  # dense output

encoded_array = ohe.fit_transform(df_model[cat_cols])
encoded_cols = ohe.get_feature_names_out(cat_cols)

encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df_model.index)

# Drop original categorical cols and attach encoded cols
df_model = pd.concat(
    [df_model.drop(columns=cat_cols), encoded_df],
    axis=1
)

df_model.head()

MemoryError: Unable to allocate 526. MiB for an array with shape (8, 8615533) and data type object

In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

X = df_model.drop(columns=['is_fraud'])
# Keep only numeric columns
X_numeric = X.select_dtypes(include=['number'])

# 1. Drop obvious IDs, keys, hashes
id_cols = ['id', 'user_id', 'card_id', 'transaction_id']  # adjust to your dataset
df_model = df_model.drop(columns=[c for c in id_cols if c in df_model.columns])


selector = VarianceThreshold(threshold=0.0)
selector.fit(X_numeric)
constant_cols = X_numeric.columns[~selector.get_support()]
df_model = df_model.drop(columns=constant_cols)

# 3. Drop columns with >90% missing values
missing_threshold = 0.9
high_na_cols = df_model.columns[df_model.isna().mean() > missing_threshold]
df_model = df_model.drop(columns=high_na_cols)

print(f"Remaining features after pre-filtering: {df_model.drop(columns=['is_fraud']).shape[1]}")
print(f"New Dataset shape: {df_model.shape}")

Remaining features after pre-filtering: 91
New Dataset shape: (8615533, 92)


In [17]:
X.head()

,mcc,num_cards_issued,credit_limit,year_pin_last_changed,current_age,retirement_age,birth_year,birth_month,per_capita_income,yearly_income,...,use_chip_chip transaction,use_chip_online transaction,use_chip_swipe transaction,card_brand_American Express,card_brand_Discovery,card_brand_Mastercard,card_brand_Visa,card_type_Credit,card_type_Debit,card_type_Prepaid
1361151,5912.0,1,198576,2015,55,67,1964,3,320202,652860,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5193004,5411.0,1,171000,2015,70,65,1949,3,381438,661032,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
5857271,5814.0,1,153000,2011,28,65,1992,1,292284,595908,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2936421,5813.0,2,260388,2007,63,66,1957,1,248490,506646,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
6978563,5812.0,1,433872,2009,51,65,1968,6,335142,683334,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [ ]:
df_model['date'] = pd.to_datetime(df_model['date'], errors='coerce')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

target = "is_fraud"
sample_size = 100_000
random_state = 42

# -----------------------------
# 1. Sample ROWS first (huge win)
# -----------------------------
if len(df_model) > sample_size:
    df_sample = df_model.sample(sample_size, random_state=random_state)
else:
    df_sample = df_model

# -----------------------------
# 2. Select numeric columns only (no copy yet)
# -----------------------------
num_cols = df_sample.select_dtypes(include=["number"]).columns.tolist()
num_cols.remove(target)

X = df_sample[num_cols]
y = df_sample[target].astype(np.int8)
date_cols = df_model['date']

# -----------------------------
# 3. Downcast numerics aggressively
# -----------------------------
for col in X.columns:
    if pd.api.types.is_float_dtype(X[col]):
        X[col] = X[col].astype(np.float32)
    elif pd.api.types.is_integer_dtype(X[col]):
        X[col] = X[col].astype(np.int32)

# -----------------------------
# 4. Clean NaN / inf IN-PLACE
# -----------------------------
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# -----------------------------
# 5. Mutual Information
# -----------------------------
mi = mutual_info_classif(
    X,
    y,
    discrete_features="auto",
    random_state=random_state,
    n_jobs=-1
)

mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
top_features = mi_series.head(60).index.tolist()

# -----------------------------
# 6. Reduce original df_model columns (no extra copy)
# -----------------------------
df_model = df_model.loc[:, top_features + [target] + date_cols]

print(f"Top {len(top_features)} features selected using Mutual Information")

C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_4556\938424569.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype(np.float32)
C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_4556\938424569.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype(np.int32)
C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_4556\938424569.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = va

Top 60 features selected using Mutual Information


In [10]:
df_model.head()

,new_city_flag,rapid_repeat_txn,user_merchant_familiarity,card_type_Debit,location_mismatch,use_chip_swipe transaction,city_changed,card_brand_Visa,card_brand_Mastercard,hour_median,...,credit_score,error_count,user_txn_count_7d,is_z_outlier,card_brand_American Express,user_avg_amount,amt_velocity_24h,user_hour_std,card_brand_Discovery,is_fraud
0,1,0,1,1.0,1,0.0,0,1.0,0.0,11.0,...,708,0,13,0,0.0,47.16,28.625954,6.905849,0.0,0
1,1,0,1,1.0,1,0.0,0,1.0,0.0,12.0,...,731,0,43,0,0.0,123.12,124.631578,6.945827,0.0,0
2,1,0,1,1.0,1,1.0,0,1.0,0.0,12.0,...,766,0,47,0,0.0,25.56,157.549290,6.986127,0.0,0
3,1,0,1,1.0,1,1.0,0,1.0,0.0,12.0,...,729,0,63,0,0.0,527.40,4.497270,6.899811,0.0,0
4,1,0,1,1.0,1,0.0,0,1.0,0.0,11.0,...,779,0,27,0,0.0,55.44,75.970778,7.016292,0.0,0


In [9]:
df_sorted = df_model.sort_values("date")

split_date = df_sorted["date"].quantile(0.8)

train_df = df_sorted[df_sorted["date"] <= split_date]
test_df  = df_sorted[df_sorted["date"] > split_date]

X_train = train_df.drop(columns=["is_fraud"])
y_train = train_df["is_fraud"]

X_test = test_df.drop(columns=["is_fraud"])
y_test = test_df["is_fraud"]

KeyError: 'date'

In [9]:
from lightgbm import LGBMClassifier
import pandas as pd

X_uni = df_model.drop(columns=['is_fraud'])
y_uni = df_model['is_fraud']

# Train LGBM
lgbm_fs = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=255,
    max_depth=-1,
    min_data_in_leaf=30,
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',
    random_state=42
)
lgbm_fs.fit(X_uni, y_uni)

# Feature importance
importance_df = pd.DataFrame({
    'feature': X_uni.columns,
    'importance': lgbm_fs.feature_importances_
}).sort_values(by='importance', ascending=False)

# Keep top 30 features
top_features_model = importance_df.head(30)['feature'].tolist()
X_selected = X_uni[top_features_model]

print("Top 30 features after LGBM:", top_features_model)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Info] Number of positive: 12816, number of negative: 8602717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.758324 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3989
[LightGBM] [Info] Number of data points in the train set: 8615533, number of used features: 60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Top 30 features after LGBM: ['merchant_id', 'mcc', 'category_amt_ratio', 'user_category_avg_amt', 'user_hour_std', 'time_since_last_txn', 'credit_score', 'user_txn_count_7d', 'abs_amount', 'yearly

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

# Base models
lgbm_base = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=255,
    max_depth=-1,
    min_data_in_leaf=30,
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',
    random_state=42
)

xgb_base = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.7,
    colsample_bytree=0.9,
    scale_pos_weight=(y_uni.value_counts()[0] / y_uni.value_counts()[1]),
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    tree_method="hist"
)

base_models = [('lgbm', lgbm_base), ('xgb', xgb_base)]

# Meta-learner
meta_learner = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=3000, class_weight='balanced'))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# Stacked classifier
stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=cv,
    n_jobs=-1,
    passthrough=False,
    stack_method='predict_proba'
)

In [ ]:
# Train stacked model
stacked_model.fit(X_selected, y_uni)
print("Stacked model trained successfully!")

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix, f1_score

# 1️⃣ Predict probabilities for the positive class
y_pred_proba = stacked_model.predict_proba(X_selected)[:, 1]

# 2️⃣ Compute ROC-AUC
roc_auc = roc_auc_score(y_uni, y_pred_proba)

# 3️⃣ Compute Precision-Recall AUC
precision, recall, thresholds = precision_recall_curve(y_uni, y_pred_proba)
pr_auc = auc(recall, precision)

# 4️⃣ Find optimal threshold (max F1-score)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = f1_scores.argmax()
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold (max F1): {optimal_threshold:.4f}")

# 5️⃣ Generate predictions using the optimal threshold
y_pred_opt = (y_pred_proba >= optimal_threshold).astype(int)

# 6️⃣ Confusion matrix
cm = confusion_matrix(y_uni, y_pred_opt)
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix:")
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

# 7️⃣ Optional: F1-score at optimal threshold
f1_opt = f1_score(y_uni, y_pred_opt)
print(f"\nROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"F1-score at optimal threshold: {f1_opt:.4f}")


In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(lgbm)

# Use a validation sample (important for speed & stability)
X_shap = X_uni.sample(20_000, random_state=42)
shap_values = explainer.shap_values(X_shap)[1]  # fraud class

shap_df = pd.DataFrame(shap_values, columns=X_shap.columns)

In [ ]:
shap_importance = (
    shap_df.abs()
    .mean()
    .sort_values(ascending=False)
)

shap_importance.head(10)


In [ ]:
corr_matrix = X_shap.corr().abs()

In [ ]:
to_drop = set()

features = shap_importance.index.tolist()

for i in range(len(features)):
    f1 = features[i]
    if f1 in to_drop:
        continue

    for j in range(i + 1, len(features)):
        f2 = features[j]
        if f2 in to_drop:
            continue

        if corr_matrix.loc[f1, f2] > 0.9:
            # Drop feature with lower SHAP importance
            if shap_importance[f1] >= shap_importance[f2]:
                to_drop.add(f2)
            else:
                to_drop.add(f1)

In [ ]:
pruned_features = [f for f in features if f not in to_drop]

print(f"Original features: {len(features)}")
print(f"Pruned features: {len(pruned_features)}")

pruned_features